# 06 Transformer, LLM, and attribute-combination extensions
This notebook records the stronger experiments requested by reviewers: domain transformers, LLM zero/few-shot, and explicit attribute combinations.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, json, re
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sentence_transformers import SentenceTransformer

DATA_DIR=Path('../data/biotools'); RESULTS_DIR=Path('../results'); RESULTS_DIR.mkdir(exist_ok=True, parents=True)
RANDOM_STATE=42; N_SPLITS=5
TRANSFORMER_MODELS = {
    'sbert_mpnet': 'sentence-transformers/all-mpnet-base-v2',
    'specter2': 'allenai/specter2_base',
    'scibert': 'allenai/scibert_scivocab_uncased',
    'codebert': 'microsoft/codebert-base',
}
LLM_3B_CANDIDATES = ['meta-llama/Llama-3.2-3B-Instruct', 'Qwen/Qwen2.5-3B-Instruct', 'microsoft/Phi-3.5-mini-instruct']


In [ ]:
def combine_text(df, cols):
    cols=[c for c in cols if c in df.columns]
    return df[cols].fillna('').astype(str).agg(' '.join, axis=1).str.replace(r'\s+', ' ', regex=True).str.strip()

single=pd.read_csv(DATA_DIR/'biotools_singlelabel.csv').fillna('')
COMBINATIONS = {
    'abstract_only': ['abstract'],
    'keywords_only': ['keywords'],
    'title_only': ['paper_title'],
    'repository_title_keywords_only': ['repo_title','keywords'],
    'abstract_readme_description': ['abstract','readme','description'],
    'abstract_repo_title_keywords': ['abstract','repo_title','keywords'],
    'description_function_text': ['description','function_text'],
}


## Frozen transformer embeddings + classical classifiers

In [ ]:
def encode_texts(texts, model_name, batch_size=32):
    model=SentenceTransformer(model_name)
    return model.encode(list(texts), batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

def eval_embedding_classifier(Xemb, y, attr, emb_name):
    skf=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows=[]
    classifiers={
        'logreg': LogisticRegression(max_iter=3000, class_weight='balanced'),
        'random_forest': RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
    }
    for clf_name, clf in classifiers.items():
        for fold,(tr,te) in enumerate(skf.split(Xemb,y),1):
            clf.fit(Xemb[tr], y[tr]); pred=clf.predict(Xemb[te])
            p,r,f,_=precision_recall_fscore_support(y[te], pred, average='macro', zero_division=0)
            rows.append({'task':'single','attribute':attr,'embedding':emb_name,'model':clf_name,'fold':fold,'precision_macro':p,'recall_macro':r,'f1_macro':f,'accuracy':accuracy_score(y[te], pred)})
    return pd.DataFrame(rows)

# Uncomment to run. It can take time and downloads model weights.
# rows=[]
# for attr, cols in COMBINATIONS.items():
#     text=combine_text(single, cols)
#     mask=text.str.len().gt(0)
#     for emb_name, model_name in TRANSFORMER_MODELS.items():
#         Xemb=encode_texts(text[mask], model_name)
#         rows.append(eval_embedding_classifier(Xemb, single.loc[mask,'primary_topic'].values, attr, emb_name))
# transformer_results=pd.concat(rows, ignore_index=True)
# transformer_results.to_csv(RESULTS_DIR/'biotools_transformer_embedding_results.csv', index=False)


## Fine-tuning plan
Use this when the frozen-embedding results justify the extra cost. Fine-tune SciBERT/SPECTER on publication fields and CodeBERT/RoBERTa on repository fields. Store the same fold IDs and evaluate with macro/micro/weighted F1.

In [ ]:
# Pseudocode placeholder for fine-tuning with Hugging Face Trainer.
# 1. Create fold assignments once and save them.
# 2. Tokenize train/test text for each attribute combination.
# 3. Fine-tune AutoModelForSequenceClassification for single-label.
# 4. For multi-label, use problem_type='multi_label_classification' and BCEWithLogitsLoss.
# 5. Write fold-level metrics and per-class classification reports to CSV.


## LLM zero-shot / few-shot evaluation template

In [ ]:
def build_zero_shot_prompt(text, labels):
    return f'''You classify bioinformatics research software into exactly one EDAM topic.
Allowed labels: {', '.join(labels)}
Return only one label.
Software text:
{text[:2500]}'''

def build_few_shot_prompt(text, labels, examples):
    ex='\n'.join([f'Text: {t[:500]}\nLabel: {y}' for t,y in examples])
    return f'''You classify bioinformatics research software into exactly one EDAM topic.
Allowed labels: {', '.join(labels)}
Examples:
{ex}
Now classify this text. Return only one label.
Text: {text[:2500]}'''

# Use a local inference stack such as Ollama/vLLM/transformers for the 3B models listed above.
# Save raw predictions to RESULTS_DIR/'biotools_llm_predictions.csv'.


## Ensemble/voting comparison

In [ ]:
# A simple, reproducible ensemble baseline over TF-IDF classifiers.
labels=sorted(single['primary_topic'].unique())
for attr, cols in COMBINATIONS.items():
    text=combine_text(single, cols); mask=text.str.len().gt(0)
    X=text[mask].values; y=single.loc[mask,'primary_topic'].values
    if len(set(y)) < 2: continue
    ensemble=VotingClassifier(estimators=[
        ('lr', Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced'))])),
        ('svm', Pipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=2, sublinear_tf=True)), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced'))])),
        ('rf', Pipeline([('tfidf', TfidfVectorizer(max_features=30000, min_df=2, sublinear_tf=True)), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1))])),
    ], voting='soft')
    # Run folds as in notebook 05; write results to CSV.
print('Template ready: run full fold loop when baseline notebook is complete.')
